# Statistical Analysis — Geography of Rights

Downstream analysis on `data/processed/master_scored.csv` (produced by
`scoring_pipeline.ipynb`):

1. **Choropleth maps** — composite index + each of the 5 dimensions (Altair).
2. **Inter-dimension correlation matrix** — how the five dimensions relate.
3. **Bivariate correlations** with predictors (Pearson + Spearman).
4. **Multivariate OLS** — composite ~ predictors, several specifications.
5. **Spearman rank regression** — robustness on ordinal structure.
6. **Findings summary**.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import altair as alt
from vega_datasets import data as vega_data

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
alt.data_transformers.disable_max_rows()

ROOT   = Path('.').resolve()
FIG    = ROOT / 'figures'
FIG.mkdir(exist_ok=True)
master = pd.read_csv(ROOT / 'data' / 'processed' / 'master_scored.csv')
print('Master shape:', master.shape)
master.head()

Master shape: (51, 74)


,State,1.1,1.2,1.3,1.4,1.5,1.6,2.1,2.2,2.3,2.4,2.5,3.1,3.2,3.3,3.4,3.5,3.6,4.1,4.2,4.3,4.4,4.5,5.1,5.2,5.3,5.4,5.5,1.1_n,1.2_n,1.3_n,1.4_n,1.5_n,1.6_n,2.1_n,2.2_n,2.3_n,2.4_n,2.5_n,3.1_n,3.2_n,3.3_n,3.4_n,3.5_n,3.6_n,4.1_n,4.2_n,4.3_n,4.4_n,4.5_n,5.1_n,5.2_n,5.3_n,5.4_n,5.5_n,Reproductive Rights,LGBTQ+ Protections,Voting Access,Labor Rights,Criminal Justice,Composite_EqualWeight,Composite_PCA,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,Rank_EqualWeight,Rank_PCA
0,Alabama,0,50,0,0,0,0,-10.25,0,0,0,0,-2,0,0,0,33,0,9.41,7.25,0,0,0,0,0,0,0.0,50,0.0,50.0,0.0,0.0,0.0,0.0,6.302521,0.0,0.0,0.0,0.0,42.857143,0.0,0.0,0.0,33.0,0.0,4.374924,16.406250,0.0,0.0,0.0,0.0,24.81203,0.0,0.0,100.0,8.333333,1.260504,12.642857,4.156235,24.962406,10.271067,9.846396,64.8,59610,59.0,63.1,26.6,4.8,0.4847,3.0,1,R,49,49
1,Alaska,100,100,100,0,0,100,8.50,50,0,50,100,0,100,100,67,67,100,53.70,13.00,0,100,100,100,33,50,0.0,0,100.0,100.0,100.0,0.0,0.0,100.0,37.815126,50.0,0.0,50.0,100.0,57.142857,100.0,100.0,67.0,67.0,100.0,58.499328,61.328125,0.0,100.0,100.0,100.0,49.62406,50.0,0.0,0.0,66.666667,47.563025,81.857143,63.965491,39.924812,59.995427,60.299966,55.5,86370,66.0,58.8,3.2,7.5,0.4220,5.2,1,R,21,21
2,Arizona,80,100,100,100,100,0,5.75,50,0,50,0,-2,0,0,33,67,0,49.54,15.15,0,100,0,0,0,50,0.0,50,80.0,100.0,100.0,100.0,100.0,0.0,33.193277,50.0,0.0,50.0,0.0,42.857143,0.0,0.0,33.0,67.0,0.0,53.415618,78.125000,0.0,100.0,0.0,0.0,24.81203,50.0,0.0,100.0,80.000000,26.638655,23.809524,46.308124,34.962406,42.343742,42.598733,52.8,72581,89.8,53.4,5.0,31.4,0.4680,3.6,1,R,24,24
3,Arkansas,0,0,0,0,0,0,-10.50,0,0,0,0,-4,0,0,0,33,0,23.22,11.00,0,0,0,0,33,0,100.0,0,0.0,0.0,0.0,0.0,0.0,0.0,5.882353,0.0,0.0,0.0,0.0,28.571429,0.0,0.0,0.0,33.0,0.0,21.251375,45.703125,0.0,0.0,0.0,0.0,49.62406,0.0,100.0,0.0,0.000000,1.176471,10.261905,13.390900,29.924812,10.950817,10.408072,65.5,56335,56.2,68.5,15.2,8.6,0.4780,3.6,1,R,48,48
4,California,80,100,100,100,100,100,45.00,100,100,100,100,6,100,100,100,67,100,85.45,16.90,100,100,100,50,67,50,100.0,0,80.0,100.0,100.0,100.0,100.0,100.0,99.159664,100.0,100.0,100.0,100.0,100.000000,100.0,100.0,100.0,67.0,100.0,97.299279,91.796875,100.0,100.0,100.0,50.0,75.18797,50.0,100.0,0.0,96.666667,99.831933,94.500000,97.819231,55.037594,88.771085,89.637992,37.7,91905,95.0,34.7,5.7,40.2,0.4890,5.3,1,D,2,2


## 1. Choropleth — Composite Index (Equal Weight)

Altair's built-in US states topojson is keyed by **FIPS code**, so we join the
composite scores on FIPS. DC has FIPS 11 and will render as a small polygon in
the DC area.

In [2]:
# State -> FIPS mapping (includes DC)
STATE_FIPS = {
    'Alabama': 1, 'Alaska': 2, 'Arizona': 4, 'Arkansas': 5, 'California': 6,
    'Colorado': 8, 'Connecticut': 9, 'Delaware': 10, 'District of Columbia': 11,
    'Florida': 12, 'Georgia': 13, 'Hawaii': 15, 'Idaho': 16, 'Illinois': 17,
    'Indiana': 18, 'Iowa': 19, 'Kansas': 20, 'Kentucky': 21, 'Louisiana': 22,
    'Maine': 23, 'Maryland': 24, 'Massachusetts': 25, 'Michigan': 26,
    'Minnesota': 27, 'Mississippi': 28, 'Missouri': 29, 'Montana': 30,
    'Nebraska': 31, 'Nevada': 32, 'New Hampshire': 33, 'New Jersey': 34,
    'New Mexico': 35, 'New York': 36, 'North Carolina': 37, 'North Dakota': 38,
    'Ohio': 39, 'Oklahoma': 40, 'Oregon': 41, 'Pennsylvania': 42,
    'Rhode Island': 44, 'South Carolina': 45, 'South Dakota': 46,
    'Tennessee': 47, 'Texas': 48, 'Utah': 49, 'Vermont': 50, 'Virginia': 51,
    'Washington': 53, 'West Virginia': 54, 'Wisconsin': 55, 'Wyoming': 56,
}
master['fips'] = master['State'].map(STATE_FIPS)
assert master['fips'].notna().all(), 'Unmapped state(s)!'

In [3]:
DIM_COLS = ['Reproductive Rights', 'LGBTQ+ Protections', 'Voting Access',
            'Labor Rights', 'Criminal Justice']

def choropleth(score_col, title):
    states_topo = alt.topo_feature(vega_data.us_10m.url, 'states')
    chart = (
        alt.Chart(states_topo)
        .mark_geoshape(stroke='white', strokeWidth=0.5)
        .encode(
            color=alt.Color(
                f'{score_col}:Q',
                scale=alt.Scale(scheme='viridis', domain=[0, 100]),
                legend=alt.Legend(title=score_col),
            ),
            tooltip=[
                alt.Tooltip('State:N'),
                alt.Tooltip(f'{score_col}:Q', format='.1f'),
            ],
        )
        .transform_lookup(
            lookup='id',
            from_=alt.LookupData(master, 'fips',
                                  ['State', score_col]),
        )
        .project('albersUsa')
        .properties(width=700, height=420, title=title)
    )
    return chart

composite_map = choropleth('Composite_EqualWeight',
                            'State-Level Human Rights Composite Index (Equal Weight)')
composite_map.save(str(FIG / 'choropleth_composite.html'))
composite_map

alt.Chart(...)

### 1a. Dimension choropleths (small multiples)

In [4]:
dim_charts = [choropleth(d, d).properties(width=340, height=210) for d in DIM_COLS]
row1 = alt.hconcat(*dim_charts[:3])
row2 = alt.hconcat(*dim_charts[3:])
dims_grid = alt.vconcat(row1, row2).resolve_scale(color='independent')
dims_grid.save(str(FIG / 'choropleth_dimensions.html'))
dims_grid

alt.VConcatChart(...)

## 2. Inter-dimension correlation matrix

Pearson correlation among the 5 dimension scores. If dimensions are highly
correlated, the composite is effectively redundant on one latent dimension
(reinforcing the PCA finding).

In [5]:
dim_corr = master[DIM_COLS].corr(method='pearson').round(3)
print(dim_corr)

                     Reproductive Rights  LGBTQ+ Protections  Voting Access  Labor Rights  Criminal Justice
Reproductive Rights                1.000               0.870          0.850         0.897             0.654
LGBTQ+ Protections                 0.870               1.000          0.852         0.849             0.800
Voting Access                      0.850               0.852          1.000         0.786             0.636
Labor Rights                       0.897               0.849          0.786         1.000             0.685
Criminal Justice                   0.654               0.800          0.636         0.685             1.000


In [6]:
dim_corr_long = (
    dim_corr.reset_index().melt(id_vars='index', var_name='Dim2', value_name='r')
    .rename(columns={'index': 'Dim1'})
)

heatmap = (
    alt.Chart(dim_corr_long)
    .mark_rect()
    .encode(
        x=alt.X('Dim1:N', sort=DIM_COLS, title=None),
        y=alt.Y('Dim2:N', sort=DIM_COLS, title=None),
        color=alt.Color('r:Q', scale=alt.Scale(scheme='redblue',
                                                domain=[-1, 1], reverse=True)),
        tooltip=['Dim1', 'Dim2', alt.Tooltip('r:Q', format='.2f')],
    )
    .properties(width=420, height=420, title='Dimension Correlations (Pearson r)')
)

labels = (
    alt.Chart(dim_corr_long).mark_text(size=13, fontWeight='bold')
    .encode(
        x=alt.X('Dim1:N', sort=DIM_COLS),
        y=alt.Y('Dim2:N', sort=DIM_COLS),
        text=alt.Text('r:Q', format='.2f'),
        color=alt.condition('abs(datum.r) > 0.6',
                            alt.value('white'), alt.value('black')),
    )
)

corr_chart = heatmap + labels
corr_chart.save(str(FIG / 'dimension_correlation_matrix.html'))
corr_chart

alt.LayerChart(...)

## 3. Predictors setup

Trifecta (P10) is categorical ∈ {D, R, Split}; encode as two dummies with
Split as reference. Everything else is numeric.

In [7]:
PRED_NUMERIC = ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9']
PRED_LABELS = {
    'P1': 'Trump Vote Share 2024 (%)',
    'P2': 'Median HH Income ($)',
    'P3': 'Percent Urban (%)',
    'P4': 'Percent NH White (%)',
    'P5': 'Percent Black (%)',
    'P6': 'Percent Hispanic (%)',
    'P7': 'Gini Coefficient',
    'P8': 'Unemployment Rate (%)',
    'P9': 'Medicaid Expansion (0/1)',
}

df = master.copy()
df = pd.concat(
    [df, pd.get_dummies(df['P10'], prefix='Trifecta', drop_first=False)],
    axis=1,
)
# Reference: Trifecta_Split (drop it), leave D and R dummies
for c in ['Trifecta_D', 'Trifecta_R']:
    if c not in df.columns:
        df[c] = 0
print('N states:', len(df), ' | Trifecta counts:', df['P10'].value_counts().to_dict())

N states: 51  | Trifecta counts: {'R': 28, 'D': 20, 'Split': 3}


## 4. Bivariate correlations — predictors vs composite & dimensions

Both Pearson (linear) and Spearman (rank) with two-sided p-values.

In [8]:
targets = ['Composite_EqualWeight'] + DIM_COLS
rows = []
for p in PRED_NUMERIC:
    for t in targets:
        x = df[p].astype(float)
        y = df[t].astype(float)
        mask = x.notna() & y.notna()
        pear = stats.pearsonr(x[mask], y[mask])
        spear = stats.spearmanr(x[mask], y[mask])
        rows.append({
            'Predictor': PRED_LABELS[p],
            'Target': t,
            'Pearson_r': round(pear.statistic, 3),
            'Pearson_p': round(pear.pvalue, 4),
            'Spearman_rho': round(spear.statistic, 3),
            'Spearman_p': round(spear.pvalue, 4),
        })
biv = pd.DataFrame(rows)
biv

,Predictor,Target,Pearson_r,Pearson_p,Spearman_rho,Spearman_p
0,Trump Vote Share 2024 (%),Composite_EqualWeight,-0.834,0.0000,-0.863,0.0000
1,Trump Vote Share 2024 (%),Reproductive Rights,-0.773,0.0000,-0.812,0.0000
2,Trump Vote Share 2024 (%),LGBTQ+ Protections,-0.827,0.0000,-0.854,0.0000
3,Trump Vote Share 2024 (%),Voting Access,-0.703,0.0000,-0.719,0.0000
4,Trump Vote Share 2024 (%),Labor Rights,-0.806,0.0000,-0.786,0.0000
5,Trump Vote Share 2024 (%),Criminal Justice,-0.689,0.0000,-0.714,0.0000
6,Median HH Income ($),Composite_EqualWeight,0.691,0.0000,0.745,0.0000
7,Median HH Income ($),Reproductive Rights,0.610,0.0000,0.637,0.0000
8,Median HH Income ($),LGBTQ+ Protections,0.702,0.0000,0.737,0.0000
9,Median HH Income ($),Voting Access,0.641,0.0000,0.654,0.0000


In [9]:
# Pivot to show just Pearson r per (predictor x target)
heat = biv.pivot(index='Predictor', columns='Target', values='Pearson_r')
# Order columns: composite first, then dimensions
heat = heat[['Composite_EqualWeight'] + DIM_COLS]
heat.round(2)

Target,Composite_EqualWeight,Reproductive Rights,LGBTQ+ Protections,Voting Access,Labor Rights,Criminal Justice
Predictor,,,,,,
Gini Coefficient,0.11,0.11,0.07,-0.04,0.24,0.10
Median HH Income ($),0.69,0.61,0.70,0.64,0.63,0.57
Medicaid Expansion (0/1),0.46,0.44,0.48,0.40,0.37,0.44
Percent Black (%),-0.16,-0.17,-0.14,-0.22,-0.10,-0.09
Percent Hispanic (%),0.28,0.28,0.25,0.26,0.27,0.18
Percent NH White (%),-0.27,-0.29,-0.24,-0.25,-0.26,-0.15
Percent Urban (%),0.46,0.42,0.47,0.35,0.46,0.41
Trump Vote Share 2024 (%),-0.83,-0.77,-0.83,-0.70,-0.81,-0.69
Unemployment Rate (%),0.40,0.40,0.34,0.33,0.42,0.30


In [10]:
heat_long = heat.reset_index().melt(id_vars='Predictor', var_name='Target', value_name='r')

biv_chart = (
    alt.Chart(heat_long)
    .mark_rect()
    .encode(
        x=alt.X('Target:N', sort=['Composite_EqualWeight'] + DIM_COLS, title=None),
        y=alt.Y('Predictor:N', sort=[PRED_LABELS[p] for p in PRED_NUMERIC], title=None),
        color=alt.Color('r:Q', scale=alt.Scale(scheme='redblue',
                                                domain=[-1, 1], reverse=True)),
        tooltip=['Predictor', 'Target', alt.Tooltip('r:Q', format='.2f')],
    )
    .properties(width=520, height=360,
                title='Bivariate Pearson r — Predictors × Rights Scores')
)
biv_labels = (
    alt.Chart(heat_long).mark_text(size=11)
    .encode(
        x=alt.X('Target:N', sort=['Composite_EqualWeight'] + DIM_COLS),
        y=alt.Y('Predictor:N', sort=[PRED_LABELS[p] for p in PRED_NUMERIC]),
        text=alt.Text('r:Q', format='.2f'),
        color=alt.condition('abs(datum.r) > 0.55',
                            alt.value('white'), alt.value('black')),
    )
)
(biv_chart + biv_labels).save(str(FIG / 'bivariate_correlations.html'))
biv_chart + biv_labels

alt.LayerChart(...)

## 5. Multivariate OLS regressions

**DV:** `Composite_EqualWeight`. We fit a sequence of specifications, from
parsimonious to fully-controlled, to see which predictors hold up.

> Sample size = 51; multicollinearity is the biggest risk (Trump share vs
> trifecta, %white vs %black, etc.). We report VIFs on the final model.

In [11]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Ensure all OLS regressors are numeric (dummy cols may land as bool)
bool_cols = ['Trifecta_D', 'Trifecta_R']
for c in bool_cols:
    df[c] = df[c].astype(int)

SPECS = {
    'M1: Political lean only':   ['P1'],
    'M2: + Income & urbanization': ['P1', 'P2', 'P3'],
    'M3: + Race composition':    ['P1', 'P2', 'P3', 'P4', 'P5', 'P6'],
    'M4: Full (adds Gini, unemp, Medicaid, trifecta)':
        ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9',
         'Trifecta_D', 'Trifecta_R'],
}

def fit_ols(dv, regressors):
    X = sm.add_constant(df[regressors].astype(float))
    y = df[dv].astype(float)
    return sm.OLS(y, X, missing='drop').fit()

models = {name: fit_ols('Composite_EqualWeight', regs) for name, regs in SPECS.items()}

for name, res in models.items():
    print('=' * 70)
    print(name)
    print('-' * 70)
    print(f'N={int(res.nobs)}  R²={res.rsquared:.3f}  Adj R²={res.rsquared_adj:.3f}  F p={res.f_pvalue:.2e}')
    coef_tbl = pd.DataFrame({
        'coef':  res.params.round(3),
        'se':    res.bse.round(3),
        't':     res.tvalues.round(2),
        'p':     res.pvalues.round(4),
    })
    print(coef_tbl)
    print()

M1: Political lean only
----------------------------------------------------------------------
N=51  R²=0.696  Adj R²=0.690  F p=2.80e-14
          coef      se     t    p
const  152.745  10.182  15.0  0.0
P1      -2.025   0.191 -10.6  0.0

M2: + Income & urbanization
----------------------------------------------------------------------
N=51  R²=0.721  Adj R²=0.704  F p=4.34e-13
          coef      se     t       p
const  100.633  31.617  3.18  0.0026
P1      -1.690   0.265 -6.37  0.0000
P2       0.001   0.000  2.03  0.0482
P3      -0.097   0.196 -0.50  0.6229

M3: + Race composition
----------------------------------------------------------------------
N=51  R²=0.816  Adj R²=0.791  F p=1.25e-14
          coef      se     t       p
const  144.438  38.030  3.80  0.0004
P1      -2.077   0.243 -8.55  0.0000
P2       0.000   0.000  0.54  0.5918
P3       0.060   0.234  0.26  0.7982
P4       0.076   0.240  0.32  0.7537
P5      -0.906   0.285 -3.18  0.0027
P6       0.055   0.346  0.16  0.875

In [12]:
# Full model detail
full = models['M4: Full (adds Gini, unemp, Medicaid, trifecta)']
print(full.summary())

                              OLS Regression Results                             
Dep. Variable:     Composite_EqualWeight   R-squared:                       0.957
Model:                               OLS   Adj. R-squared:                  0.945
Method:                    Least Squares   F-statistic:                     79.38
Date:                   Thu, 16 Apr 2026   Prob (F-statistic):           3.66e-23
Time:                           12:02:39   Log-Likelihood:                -164.57
No. Observations:                     51   AIC:                             353.1
Df Residuals:                         39   BIC:                             376.3
Df Model:                             11                                         
Covariance Type:               nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         52.4657 

In [13]:
# VIF on the full model's regressors
X_full = sm.add_constant(df[SPECS['M4: Full (adds Gini, unemp, Medicaid, trifecta)']].astype(float)).dropna()
vifs = pd.DataFrame({
    'Variable': X_full.columns,
    'VIF': [variance_inflation_factor(X_full.values, i) for i in range(X_full.shape[1])],
}).round(2)
vifs

,Variable,VIF
0,const,1979.80
1,P1,6.44
2,P2,4.19
3,P3,3.86
4,P4,4.24
5,P5,4.01
6,P6,3.92
7,P7,3.01
8,P8,1.87
9,P9,1.44


### 5a. OLS coefficient plot — preferred model (M2)

M2 (political lean + income + urbanization) is the parsimonious baseline. We
plot standardized coefficients (β on z-scored X) so effect sizes are comparable.

In [14]:
def standardized_ols(dv, regressors):
    z = df[regressors + [dv]].dropna().copy()
    for c in regressors + [dv]:
        z[c] = (z[c] - z[c].mean()) / z[c].std()
    X = sm.add_constant(z[regressors])
    y = z[dv]
    return sm.OLS(y, X).fit()

m2_std = standardized_ols('Composite_EqualWeight', SPECS['M2: + Income & urbanization'])
coef_plot_df = pd.DataFrame({
    'predictor': [PRED_LABELS[p] for p in SPECS['M2: + Income & urbanization']],
    'beta':      m2_std.params.drop('const').values,
    'ci_lo':     m2_std.conf_int().drop('const')[0].values,
    'ci_hi':     m2_std.conf_int().drop('const')[1].values,
})

base = alt.Chart(coef_plot_df).encode(
    y=alt.Y('predictor:N', sort='-x', title=None),
)
bars = base.mark_bar().encode(
    x=alt.X('beta:Q', title='Standardized coefficient (β)'),
    color=alt.condition('datum.beta > 0',
                        alt.value('#2166ac'), alt.value('#b2182b')),
)
err = base.mark_rule(strokeWidth=2).encode(x='ci_lo:Q', x2='ci_hi:Q')
zero = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(strokeDash=[4, 4]).encode(x='x:Q')

coef_chart = (bars + err + zero).properties(
    width=520, height=160,
    title='M2 — Standardized OLS β (95% CI): Composite ~ Political + Income + Urban'
)
coef_chart.save(str(FIG / 'ols_m2_coefficients.html'))
coef_chart

alt.LayerChart(...)

## 6. Spearman rank regressions — robustness

Convert each variable to its rank, then re-fit the same specs. If the sign and
significance pattern on **political lean** survives, the result isn't an artifact
of the raw distributions (right-skew on income, bimodal on trifecta dummies, etc.).

In [15]:
def rank_df(cols):
    r = df[cols].copy()
    for c in cols:
        r[c] = r[c].rank()
    return r

spearman_results = {}
for name, regs in SPECS.items():
    cols = regs + ['Composite_EqualWeight']
    r = rank_df(cols)
    X = sm.add_constant(r[regs])
    y = r['Composite_EqualWeight']
    res = sm.OLS(y, X, missing='drop').fit()
    spearman_results[name] = res
    print(f'{name}:  R²={res.rsquared:.3f}  p={res.f_pvalue:.2e}')

# Show the M2 rank-regression coefficients
print()
print('M2 (rank):')
res = spearman_results['M2: + Income & urbanization']
print(pd.DataFrame({
    'coef': res.params.round(3),
    'p':    res.pvalues.round(4),
}))

M1: Political lean only:  R²=0.745  p=3.73e-16
M2: + Income & urbanization:  R²=0.788  p=7.45e-16
M3: + Race composition:  R²=0.835  p=1.25e-15
M4: Full (adds Gini, unemp, Medicaid, trifecta):  R²=0.910  p=6.07e-17

M2 (rank):
         coef       p
const  36.425  0.0000
P1     -0.674  0.0000
P2      0.291  0.0044
P3     -0.018  0.8379


## 7. Findings summary

> The block below prints a plain-text digest of the quantitative findings you
> can paste (lightly edited) into the report's Statistical Analysis section.

In [16]:
lines = []
lines.append('FINDINGS — Geography of Rights, preliminary statistical results')
lines.append('=' * 64)
lines.append(f'N = {len(df)} (50 states + DC)')
lines.append('')
lines.append('[1] Inter-dimension correlations (Pearson r)')
lines.append('    Minimum r among the 5 dimensions: '
             f'{dim_corr.where(~np.eye(5, dtype=bool)).min().min():.2f}')
lines.append(f'    Maximum r among the 5 dimensions: '
             f'{dim_corr.where(~np.eye(5, dtype=bool)).max().max():.2f}')
lines.append('    All dimensions load positively on the same latent axis.')
lines.append('')
lines.append('[2] Strongest bivariate predictors of the composite index (|r| desc)')
comp_only = biv[biv['Target'] == 'Composite_EqualWeight'].copy()
comp_only['abs_r'] = comp_only['Pearson_r'].abs()
for _, r in comp_only.sort_values('abs_r', ascending=False).head(5).iterrows():
    lines.append(f"    {r['Predictor']:<32s}  r={r['Pearson_r']:+.2f}  "
                 f"rho={r['Spearman_rho']:+.2f}  p={r['Pearson_p']:.4f}")
lines.append('')
lines.append('[3] OLS specifications — R² ladder')
for name, res in models.items():
    lines.append(f"    {name:<55s}  R²={res.rsquared:.3f}  adj={res.rsquared_adj:.3f}")
lines.append('')
lines.append('[4] M2 preferred spec (parsimonious): '
             'Composite ~ TrumpShare + Income + %Urban')
m2 = models['M2: + Income & urbanization']
for v in ['P1', 'P2', 'P3']:
    lines.append(f"    {PRED_LABELS[v]:<32s}  β={m2.params[v]:+.4f}  p={m2.pvalues[v]:.4f}")
lines.append('')
lines.append('[5] Rank-based robustness (Spearman regression) — M2')
m2r = spearman_results['M2: + Income & urbanization']
for v in ['P1', 'P2', 'P3']:
    lines.append(f"    {PRED_LABELS[v]:<32s}  β={m2r.params[v]:+.4f}  p={m2r.pvalues[v]:.4f}")
print('\n'.join(lines))

# Also write to disk for report drafting
(FIG.parent / 'findings_summary.txt').write_text('\n'.join(lines))
print('\nWrote findings_summary.txt')

FINDINGS — Geography of Rights, preliminary statistical results
N = 51 (50 states + DC)

[1] Inter-dimension correlations (Pearson r)
    Minimum r among the 5 dimensions: 0.64
    Maximum r among the 5 dimensions: 0.90
    All dimensions load positively on the same latent axis.

[2] Strongest bivariate predictors of the composite index (|r| desc)
    Trump Vote Share 2024 (%)         r=-0.83  rho=-0.86  p=0.0000
    Median HH Income ($)              r=+0.69  rho=+0.74  p=0.0000
    Medicaid Expansion (0/1)          r=+0.46  rho=+0.49  p=0.0006
    Percent Urban (%)                 r=+0.46  rho=+0.56  p=0.0007
    Unemployment Rate (%)             r=+0.40  rho=+0.34  p=0.0038

[3] OLS specifications — R² ladder
    M1: Political lean only                                  R²=0.696  adj=0.690
    M2: + Income & urbanization                              R²=0.721  adj=0.704
    M3: + Race composition                                   R²=0.816  adj=0.791
    M4: Full (adds Gini, unemp, Medi